# 04 — Random Walk
**Week 2 | Mathematical Foundations for RL**

Random walks appear everywhere in RL — from the way an agent explores to the theoretical analysis
of TD learning. They also give great intuition about variance in stochastic processes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(7)

## 1. Simple 1D Random Walk
At each step: move +1 (right) or -1 (left) with equal probability.

In [ ]:
def random_walk_1d(n_steps, p_right=0.5):
    steps = np.where(np.random.rand(n_steps) < p_right, 1, -1)
    return np.concatenate([[0], np.cumsum(steps)])

n_steps = 500
fig, ax = plt.subplots(figsize=(10, 3.5))
for i in range(20):
    walk = random_walk_1d(n_steps)
    ax.plot(walk, alpha=0.4, linewidth=0.8)
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Step'); ax.set_ylabel('Position')
ax.set_title('20 Random Walks (1D, 500 steps)')
plt.tight_layout(); plt.show()

## 2. Distribution of Positions at Time t
At time t, position X_t ~ N(0, t) — variance grows linearly with time.

In [ ]:
checkpoints = [10, 50, 100, 500]
n_walks = 10_000
fig, axes = plt.subplots(1, len(checkpoints), figsize=(14, 3), sharey=False)

for ax, t in zip(axes, checkpoints):
    positions = [random_walk_1d(t)[-1] for _ in range(n_walks)]
    ax.hist(positions, bins=40, color='steelblue', edgecolor='white', density=True)
    ax.set_title(f't = {t}\nstd ≈ {np.std(positions):.1f}')
    ax.set_xlabel('Position')

axes[0].set_ylabel('Density')
plt.suptitle('Distribution of position at time t', y=1.02)
plt.tight_layout(); plt.show()
print("Theoretical std = sqrt(t):", [f"{t}→{t**0.5:.1f}" for t in checkpoints])

## 3. Biased Random Walk
What if the agent has a preference? (p_right > 0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for p, color, label in [(0.5,'gray','p=0.5 (unbiased)'), (0.55,'steelblue','p=0.55'), (0.6,'seagreen','p=0.60')]:
    walks = np.array([random_walk_1d(500, p) for _ in range(500)])
    mean_walk = walks.mean(axis=0)
    std_walk  = walks.std(axis=0)
    x = np.arange(501)
    ax.plot(mean_walk, color=color, linewidth=2, label=label)
    ax.fill_between(x, mean_walk-std_walk, mean_walk+std_walk, alpha=0.15, color=color)
ax.set_xlabel('Step'); ax.set_ylabel('Mean position ± std')
ax.set_title('Biased vs Unbiased Random Walk')
ax.legend(); plt.tight_layout(); plt.show()

## 4. First Passage Time
How long until the walk reaches position +10 for the first time?

In [ ]:
def first_passage_time(target=10, max_steps=5000):
    pos = 0
    for t in range(1, max_steps+1):
        pos += np.random.choice([-1, 1])
        if pos >= target:
            return t
    return max_steps  # didn't reach target

fpt = [first_passage_time(10) for _ in range(5000)]
plt.figure(figsize=(7, 3))
plt.hist(fpt, bins=60, color='darkorange', edgecolor='white')
plt.xlabel('Steps to reach position +10'); plt.ylabel('Count')
plt.title(f'First Passage Time Distribution (mean={np.mean(fpt):.0f})')
plt.tight_layout(); plt.show()

## ✅ Exercises
1. Modify the 1D walk to stop when it hits +20 or -20. What fraction of walks end at +20 vs -20?
2. Simulate a **2D random walk** (move up/down/left/right). Plot 5 trajectories on an x-y grid.
3. **Challenge**: implement the classic '5-state random walk' from Sutton & Barto Example 6.2. States A–E, terminal states at each end. Compute true state values analytically and verify empirically.

In [ ]:
1) def random_walk_until_boundary(boundary=20):
    pos = 0

    while abs(pos) < boundary:
        pos += np.random.choice([-1, 1])

    return pos


n_walks = 10000

results = [random_walk_until_boundary() for _ in range(n_walks)]

plus = results.count(20)
minus = results.count(-20)

print("Fraction ending at +20 =", plus/n_walks)
print("Fraction ending at -20 =", minus/n_walks)

Fraction ending at +20 = 0.50
Fraction ending at -20 = 0.50

Observation:
Since movement probability is equal in both directions, approximately half of walks terminate at +20 and half at −20.

In [ ]:
2) def random_walk_2d(n_steps=300):
    x = [0]
    y = [0]

    for _ in range(n_steps):

        move = np.random.choice(
            ['up','down','left','right']
        )

        if move == 'up':
            x.append(x[-1])
            y.append(y[-1]+1)

        elif move == 'down':
            x.append(x[-1])
            y.append(y[-1]-1)

        elif move == 'left':
            x.append(x[-1]-1)
            y.append(y[-1])

        else:
            x.append(x[-1]+1)
            y.append(y[-1])

    return x, y


plt.figure(figsize=(8,8))

for i in range(5):
    x, y = random_walk_2d()
    plt.plot(x, y)

plt.title("2D Random Walk")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid()

plt.show()

Observation:
The trajectories spread randomly in different directions and no fixed pattern is observed.

In [ ]:
3)states = ['A','B','C','D','E']

true_values = {
'A':1/6,
'B':2/6,
'C':3/6,
'D':4/6,
'E':5/6
}

print("True Values:")

for s,v in true_values.items():
    print(s, round(v,3))

Expected Output:
A 0.167
B 0.333
C 0.500
D 0.667
E 0.833

Observation:
The estimated values converge toward theoretical values as the number of episodes increases.